In [ ]:

!apt-get install openjdk-8-jdk-headless -qq > /dev/null


!rm -f kafka_2.13-3.3.1.tgz
!wget -O kafka_2.13-3.3.1.tgz https://archive.apache.org/dist/kafka/3.3.1/kafka_2.13-3.3.1.tgz
!tar -xzf kafka_2.13-3.3.1.tgz


!pip install -q pyspark==3.4.1 kafka-python-ng

In [ ]:
import os
import subprocess
import time


KAFKA_DIR = "./kafka_2.13-3.3.1"
ZOOKEEPER_CONFIG = os.path.join(KAFKA_DIR, "config", "zookeeper.properties")
KAFKA_SERVER_CONFIG = os.path.join(KAFKA_DIR, "config", "server.properties")


!killall -q java
time.sleep(2)


with open(KAFKA_SERVER_CONFIG, 'r') as f:
    lines = f.readlines()


new_lines = []
listener_configured = False
advertised_listener_configured = False
for line in lines:
    if line.strip().startswith('listeners=') or line.strip().startswith('advertised.listeners='):
        continue
    new_lines.append(line)


new_lines.append('listeners=PLAINTEXT://127.0.0.1:9092\n')
new_lines.append('advertised.listeners=PLAINTEXT://127.0.0.1:9092\n')


with open(KAFKA_SERVER_CONFIG, 'w') as f:
    f.writelines(new_lines)

print(f"Modified {KAFKA_SERVER_CONFIG} to listen on 127.0.0.1:9092")


zookeeper_cmd = f"{KAFKA_DIR}/bin/zookeeper-server-start.sh {ZOOKEEPER_CONFIG}"
print(f"Starting Zookeeper with command: {zookeeper_cmd}")
zookeeper_process = subprocess.Popen(zookeeper_cmd.split())
time.sleep(5)
print("Zookeeper started.")


kafka_cmd = f"{KAFKA_DIR}/bin/kafka-server-start.sh {KAFKA_SERVER_CONFIG}"
print(f"Starting Kafka with command: {kafka_cmd}")
kafka_process = subprocess.Popen(kafka_cmd.split())
time.sleep(25)
print("Kafka started.")

print("Zookeeper-ը և Kafka-ն հաջողությամբ միացված են:")

In [ ]:
import json
import random
import sqlite3
import threading
import time
from kafka import KafkaProducer
from kafka.errors import NoBrokersAvailable


conn = sqlite3.connect('football_stats.db')
cursor = conn.cursor()
cursor.execute('''
    CREATE TABLE IF NOT EXISTS team_stats (
        team TEXT PRIMARY KEY,
        shots INTEGER,
        fouls INTEGER
    )
''')

cursor.execute("INSERT OR IGNORE INTO team_stats VALUES ('Real Madrid', 0, 0)")
cursor.execute("INSERT OR IGNORE INTO team_stats VALUES ('Barcelona', 0, 0)")
conn.commit()
conn.close()

def run_producer():
    producer = None
    max_retries = 10
    retry_delay = 5

    for i in range(max_retries):
        try:
            print(f"Producer: Attempting to connect to Kafka (Attempt {i+1}/{max_retries})...")
            producer = KafkaProducer(
                bootstrap_servers=['127.0.0.1:9092'],
                value_serializer=lambda v: json.dumps(v).encode('utf-8')
            )
            print("Producer: Successfully connected to Kafka.")
            break
        except NoBrokersAvailable:
            print(f"Producer: Kafka brokers not available. Retrying in {retry_delay} seconds...")
            time.sleep(retry_delay)
        except Exception as e:
            print(f"Producer: An unexpected error occurred during Kafka connection: {e}")
            time.sleep(retry_delay)

    if producer is None:
        print("Producer: Failed to connect to Kafka after multiple retries. Exiting producer.")
        return

    teams = ['Real Madrid', 'Barcelona']
    players = {
        'Real Madrid': ['Vinicius Jr', 'Mbappe', 'Bellingham'],
        'Barcelona': ['Lamine Yamal', 'Lewandowski', 'Raphinha']
    }
    event_types = ['Shot', 'Foul', 'Pass']

    print("Producer-ը սկսեց տվյալների հոսքը...")

    for minute in range(1, 61):
        team = random.choice(teams)
        player = random.choice(players[team])
        event = random.choice(event_types)

        payload = {'minute': minute, 'team': team, 'player': player, 'event': event}
        producer.send('football-events', value=payload)
        time.sleep(0.5)
    producer.flush()
    print("Producer-ը ավարտեց աշխատանքը:")


threading.Thread(target=run_producer).start()

In [ ]:
import time
import json
import sqlite3
from kafka import KafkaConsumer
from kafka.errors import NoBrokersAvailable

def run_spark_simulation():
    consumer = None
    max_retries = 10
    retry_delay = 5

    for i in range(max_retries):
        try:
            print(f"Attempting to connect to Kafka (Attempt {i+1}/{max_retries})...")
            consumer = KafkaConsumer(
                'football-events',
                bootstrap_servers=['127.0.0.1:9092'],
                auto_offset_reset='earliest',
                value_deserializer=lambda x: json.loads(x.decode('utf-8')),
                consumer_timeout_ms=5000
            )
            print("Successfully connected to Kafka.")
            break
        except NoBrokersAvailable:
            print(f"Kafka brokers not available. Retrying in {retry_delay} seconds...")
            time.sleep(retry_delay)
        except Exception as e:
            print(f"An unexpected error occurred during Kafka connection: {e}")
            time.sleep(retry_delay)

    if consumer is None:
        print("Failed to connect to Kafka after multiple retries. Exiting simulation.")
        return

    print("Spark-ը միացավ Kafka-ին և մշակում է տվյալները...")

    for message in consumer:
        event_data = message.value
        team = event_data['team']
        event = event_data['event']


        db = sqlite3.connect('football_stats.db')
        cur = db.cursor()

        if event == 'Shot':
            cur.execute("UPDATE team_stats SET shots = shots + 1 WHERE team = ?", (team,))
        elif event == 'Foul':
            cur.execute("UPDATE team_stats SET fouls = fouls + 1 WHERE team = ?", (team,))

        db.commit()
        db.close()

run_spark_simulation()

In [ ]:
import IPython
from google.colab import output


def get_live_data():
    conn = sqlite3.connect('football_stats.db')
    cursor = conn.cursor()
    cursor.execute("SELECT team, shots, fouls FROM team_stats")
    data = cursor.fetchall()
    conn.close()
    return IPython.display.JSON({row[0]: {'shots': row[1], 'fouls': row[2]} for row in data})


output.register_callback('notebook.get_live_data', get_live_data)

html_code = """
<div style="font-family: Arial, sans-serif; text-align: center; background: #1e1e2e; color: white; padding: 20px; border-radius: 10px;">
    <h2>🏆 LIVE Football Analytics Dashboard 🏆</h2>
    <hr style="border-color: #444;">
    <div style="display: flex; justify-content: space-around; margin-top: 20px;">
        <div style="background: #e74c3c; padding: 20px; border-radius: 8px; width: 40%;">
            <h3>Real Madrid</h3>
            <p id="rm-shots" style="font-size: 24px; font-weight: bold;">Shots: 0</p>
            <p id="rm-fouls" style="font-size: 18px; color: #ddd;">Fouls: 0</p>
        </div>
        <div style="background: #3498db; padding: 20px; border-radius: 8px; width: 40%;">
            <h3>Barcelona</h3>
            <p id="barca-shots" style="font-size: 24px; font-weight: bold;">Shots: 0</p>
            <p id="barca-fouls" style="font-size: 18px; color: #ddd;">Fouls: 0</p>
        </div>
    </div>
</div>

<script>
    async function updateDashboard() {

        const response = await google.colab.kernel.invokeFunction('notebook.get_live_data', [], {});
        const data = response.data['application/json'];


        document.getElementById('rm-shots').innerText = "Shots: " + data['Real Madrid']['shots'];
        document.getElementById('rm-fouls').innerText = "Fouls: " + data['Real Madrid']['fouls'];

        document.getElementById('barca-shots').innerText = "Shots: " + data['Barcelona']['shots'];
        document.getElementById('barca-fouls').innerText = "Fouls: " + data['Barcelona']['fouls'];
    }


    setInterval(updateDashboard, 1000);
</script>
"""


IPython.display.display(IPython.display.HTML(html_code))